# Phase 1: SIR Measurement on GCR Baseline

**Goal:** Measure how permissive GCR's KG-Trie is using the Semantic Irrelevance
Ratio (SIR). This generates the baseline numbers for Chapter 4 that motivate DCA-Trie.

**Runs on:** CPU or GPU (auto-detects CUDA). MiniLM is fast on CPU too.

**What you'll get:**
- SIR table by hop depth (1/2/3/4)
- Average trie size per hop depth
- SIR vs. hop depth plot saved to `figures/gcr_baseline_sir.png`
- Results exported to `data/gcr_baseline_sir_results.json`

## 1. Colab / Local Environment Setup

In [ ]:
import sys, os, json, warnings, gc, time, textwrap, pprint, copy
import itertools, collections, random, math, re
import typing, pathlib, hashlib, functools
import numpy as np
from tqdm import tqdm
from collections import defaultdict

IN_COLAB = 'google.colab' in sys.modules
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Running in Colab: {IN_COLAB}")

if IN_COLAB:
    if not os.path.exists('dca-trie'):
        !git clone https://github.com/Adjanour/dca-trie.git
        %cd dca-trie
        !git submodule update --init
    else:
        %cd dca-trie
    !pip install -q -e .
else:
    print("Running locally. Ensure poetry env is active:")
    print("  source $(poetry env info --path)/bin/activate")

print("\nSetup complete.")

## 2. Set HuggingFace Token

The GCR tokenizer is gated. Get your token at https://huggingface.co/settings/tokens
and request access at https://huggingface.co/rmanluo/GCR-Meta-Llama-3.1-8B-Instruct

In [ ]:
from huggingface_hub import login

HF_TOKEN = "" or os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True)
    print("HF_TOKEN configured.")
else:
    print("WARNING: No HF_TOKEN set. Gated model download may fail.")

## 3. Imports

In [ ]:
from dca_trie.semantic_scorer import SemanticScorer
from dca_trie.sir_measurement import SIRMeasurer
from dca_trie.mid_resolver import MidResolver

import gcr.src.utils as gcr_utils
from gcr.src.utils.graph_utils import build_graph, dfs, get_truth_paths
from gcr.src.trie import MarisaTrie

from datasets import load_dataset
from transformers import AutoTokenizer

%load_ext autoreload
%autoreload 2
print("All imports OK.")

## 4. Initialize Semantic Scorer

Uses `all-MiniLM-L6-v2` (384-dim, ~80MB). Auto-detects GPU if available.

In [ ]:
scorer = SemanticScorer()
print(f"Device: {scorer.device}")

test_score = scorer.score(
    "Barack Obama -> people.person.spouse -> Michelle Obama",
    "Who is the spouse of Barack Obama?"
)
print(f"Smoke test score: {test_score:.4f}")
assert 0.0 < test_score < 1.0

## 5. Build MID Resolver

WebQSP graphs contain Freebase MIDs (e.g. `m.0c6q0` for "Warsaw").
MiniLM needs readable names. MID resolver extracts name mappings

In [ ]:
resolver = MidResolver(cache_path="data/mid_to_name.json")
sample = load_dataset("rmanluo/RoG-webqsp", split="test[:50]")
resolver.build_from_dataset(sample)

# Optionally download Freebase names for ~99% coverage
# resolver.download_and_build_fb_names(target_path="data/fb_entity_names.txt.gz")

cov = resolver.coverage(sample)
print(f"MID coverage: {cov['coverage_pct']}% ({cov['resolved']}/{cov['total_mids']})")

print("\nSample resolutions:")
for mid, name in list(resolver.mid_to_name.items())[:10]:
    print(f"  {mid} -> {name}")

## 6. Patch GCR's path_to_string with MID Resolution

Monkey-patch so every path string has readable names before MiniLM scores it.

In [ ]:
_ORIG_PATH_TO_STR = gcr_utils.path_to_string

def _resolved_path_to_string(path):
    raw = _ORIG_PATH_TO_STR(path)
    return resolver.resolve_path(raw)

gcr_utils.path_to_string = _resolved_path_to_string

# Verify
g = build_graph(sample[0]["graph"])
test_paths = dfs(g, sample[0]["q_entity"], 2)
if test_paths:
    print(f"Raw: {_ORIG_PATH_TO_STR(test_paths[0])}")
    print(f"Resolved: {gcr_utils.path_to_string(test_paths[0])}")

## 7. Load WebQSP Data

In [ ]:
NUM_QUESTIONS = 100
split = f"test[:{NUM_QUESTIONS}]" if NUM_QUESTIONS > 0 else "test"
dataset = load_dataset("rmanluo/RoG-webqsp", split=split)
questions = list(dataset)
print(f"Loaded {len(questions)} questions from WebQSP")
print(f"Sample: {questions[0]['question']}")

print("\nLoading GCR tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct",
    trust_remote_code=True,
)
print(f"Vocab size: {len(tokenizer)}")

## 8. Build GCR-Style Tries

Replicates GCR's `get_graph_index()`: DFS from q_entity, tokenize, build MarisaTrie.

In [ ]:
# GCR default is 2. Increase to 3-4 for richer per-hop SIR analysis.
MAX_HOPS = 2

def build_gcr_trie(question_dict, tokenizer, max_hops=MAX_HOPS):
    g = build_graph(question_dict["graph"])
    paths_list = dfs(g, question_dict["q_entity"], max_hops)
    if not paths_list:
        return None, []

    paths_list_str = [gcr_utils.path_to_string(p) for p in paths_list]
    tokenized_paths = tokenizer(
        paths_list_str, padding=False, add_special_tokens=False
    ).input_ids
    tokenized_path_list = [
        ids + [tokenizer.eos_token_id] for ids in tokenized_paths
    ]
    trie = MarisaTrie(tokenized_path_list, max_token_id=len(tokenizer) + 1)
    return trie, paths_list_str


sample_trie, sample_paths = build_gcr_trie(questions[0], tokenizer)
print(f"Q: {questions[0]['question']}")
print(f"Paths in trie: {len(sample_trie)}")
print("\nFirst 3 paths:")
for p in sample_paths[:3]:
    print(f"  {p}")

## 9. Measure Baseline SIR

For each question: build GCR trie, measure overall SIR and per-hop SIR.

In [ ]:
measurer = SIRMeasurer(scorer, tokenizer)

per_question_results = []

for i, data in enumerate(tqdm(questions, desc="Measuring SIR")):
    trie, path_strs = build_gcr_trie(data, tokenizer)
    if trie is None:
        continue

    sir_result = measurer.measure_from_trie(trie, data["question"])
    hop_sir = measurer.measure_per_hop(trie, data["question"])

    per_question_results.append({
        "id": data.get("id", i),
        "question": data["question"],
        "num_paths": sir_result["num_paths"],
        "sir": sir_result["sir"],
        "max_sim": sir_result["max_similarity"],
        "avg_sim": sir_result["avg_similarity"],
        "hop_sir": hop_sir,
    })

print(f"\nMeasured SIR for {len(per_question_results)} questions")
for r in per_question_results[:5]:
    print(f"\n[{r['id']}] {r['question'][:70]}")
    print(f"  SIR: {r['sir']:.4f}  |  Paths: {r['num_paths']}  |  MaxSim: {r['max_sim']:.4f}")

## 10. Results Table

In [ ]:
print("=" * 72)
print(f"{'Metric':<30} {'Mean':<12} {'Min':<12} {'Max':<12}")
print("-" * 72)

valid = [r for r in per_question_results if r["sir"] is not None]
all_sirs = [r["sir"] for r in valid]
all_path_counts = [r["num_paths"] for r in valid]
all_max_sims = [r["max_sim"] for r in valid]

print(f"{'SIR':<30} {np.mean(all_sirs):<12.4f} {np.min(all_sirs):<12.4f} {np.max(all_sirs):<12.4f}")
print(f"{'Trie size':<30} {np.mean(all_path_counts):<12.1f} {np.min(all_path_counts):<12.0f} {np.max(all_path_counts):<12.0f}")
print(f"{'Max similarity':<30} {np.mean(all_max_sims):<12.4f} {np.min(all_max_sims):<12.4f} {np.max(all_max_sims):<12.4f}")
print("=" * 72)

print("\n" + "=" * 72)
print("SIR by Hop Depth")
print("=" * 72)
print(f"{'Hop':<8} {'Questions':<12} {'Avg Paths':<12} {'SIR':<12}")
print("-" * 72)

all_hop_data = {h: {"sirs": [], "counts": []} for h in [1, 2, 3, 4]}
for r in per_question_results:
    for hop, hop_data in r["hop_sir"].items():
        if hop_data["num_paths"] > 0 and hop_data["sir"] is not None:
            all_hop_data[hop]["sirs"].append(hop_data["sir"])
            all_hop_data[hop]["counts"].append(hop_data["num_paths"])

for hop in [1, 2, 3, 4]:
    d = all_hop_data[hop]
    if d["sirs"]:
        print(f"{hop:<8} {len(d['sirs']):<12} {np.mean(d['counts']):<12.1f} {np.mean(d['sirs']):<12.4f}")
    else:
        print(f"{hop:<8} {'N/A':<12} {'N/A':<12} {'N/A':<12}")

## 11. SIR vs. Hop Depth Plot

In [ ]:
import matplotlib.pyplot as plt

hop_depths = []
mean_sirs = []
mean_trie_sizes = []

for hop in [1, 2, 3, 4]:
    d = all_hop_data[hop]
    if d["sirs"]:
        hop_depths.append(hop)
        mean_sirs.append(np.mean(d["sirs"]))
        mean_trie_sizes.append(np.mean(d["counts"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(hop_depths, mean_sirs, "bo-", linewidth=2, markersize=8)
ax1.set_xlabel("Hop Depth", fontsize=12)
ax1.set_ylabel("Mean SIR", fontsize=12)
ax1.set_title("GCR Baseline: SIR Increases with Hop Depth", fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1)

ax2.bar(hop_depths, mean_trie_sizes, color="orange", alpha=0.7)
ax2.set_xlabel("Hop Depth", fontsize=12)
ax2.set_ylabel("Avg Trie Size (paths)", fontsize=12)
ax2.set_title("GCR Baseline: Trie Size at Each Hop", fontsize=13)
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/gcr_baseline_sir.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to figures/gcr_baseline_sir.png")

## 12. Export Results

In [ ]:
output = {
    "dataset": "RoG-webqsp",
    "num_questions": len(per_question_results),
    "model": "GCR-Meta-Llama-3.1-8B-Instruct",
    "index_path_length": MAX_HOPS,
    "overall": {
        "mean_sir": float(np.mean(all_sirs)),
        "median_sir": float(np.median(all_sirs)),
        "mean_trie_size": float(np.mean(all_path_counts)),
        "median_trie_size": float(np.median(all_path_counts)),
    },
    "per_hop": {},
    "per_question": per_question_results,
}

for hop in [1, 2, 3, 4]:
    d = all_hop_data[hop]
    if d["sirs"]:
        output["per_hop"][str(hop)] = {
            "mean_sir": float(np.mean(d["sirs"])),
            "mean_trie_size": float(np.mean(d["counts"])),
            "num_questions": len(d["sirs"]),
        }
    else:
        output["per_hop"][str(hop)] = {
            "mean_sir": None,
            "mean_trie_size": None,
            "num_questions": 0,
        }

os.makedirs("data", exist_ok=True)
with open("data/gcr_baseline_sir_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved to data/gcr_baseline_sir_results.json")
print(f"\nMean SIR: {output['overall']['mean_sir']:.4f}")
print(f"Mean trie size: {output['overall']['mean_trie_size']:.1f}")
for k, v in output["per_hop"].items():
    if v["mean_sir"]:
        print(f"  Hop {k}: SIR={v['mean_sir']:.4f}, paths={v['mean_trie_size']:.1f}, n={v['num_questions']}")

## Phase 1 Exit Criteria
- [ ] SIR measurement runs without errors
- [ ] Produces per-question and per-hop SIR values
- [ ] SIR shows clear upward trend with hop depth
- [ ] Average trie size per step is reported
- [ ] Plot saved to `figures/gcr_baseline_sir.png`
- [ ] Results exported to `data/gcr_baseline_sir_results.json`